# Indices de lexicalité des OCR non natifs

In [ ]:
# Installations
!pip install fasttext wordfreq psycopg2-binary duckdb


In [ ]:
import re
import unicodedata
import os
import urllib.request
import csv
import fasttext
import numpy as np
import duckdb
import pandas as pd
from wordfreq import zipf_frequency
from multiprocessing import Pool, cpu_count
from collections import Counter


In [ ]:
# Téléchargement du modèle fastText (si absent)
MODEL_PATH = "lid.176.ftz"

if not os.path.exists(MODEL_PATH):
    print("Téléchargement du modèle fastText...")
    url = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz"
    urllib.request.urlretrieve(url, MODEL_PATH)

taille = os.path.getsize(MODEL_PATH)
print(f"Taille du fichier : {taille:,} octets")
if taille < 900_000:
    raise ValueError(f"Fichier corrompu ({taille} octets) — supprime-le et relance")

print("Modèle prêt")


# Procédure
- Tokenisation (extraction des mots alphabétiques)
- Normalisation (minuscules, accents)
- Détection de langue via fastText — seuls les docs en anglais sont analysés
- Indice de lexicalité = mots valides (Zipf ≥ 2.5) / total mots


In [ ]:
def tokenize(text):
    """Extrait uniquement les mots alphabétiques."""
    return re.findall(r"\b[a-zA-Z]+\b", text)

def normalize_text(text):
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def analyze_text(text, threshold=2.5, lang="en", top_n=20):
    """Calcule l'indice de lexicalité d'un texte."""
    normalized = normalize_text(text)
    tokens = tokenize(normalized)
    total_words = len(tokens)

    valid_words, invalid_words = [], []
    for word in tokens:
        freq = zipf_frequency(word, lang)
        if freq >= threshold:
            valid_words.append(freq)
        else:
            invalid_words.append(word)

    valid_count = len(valid_words)
    lexicality = (valid_count / total_words * 100) if total_words else 0
    avg_zipf = sum(valid_words) / len(valid_words) if valid_words else 0
    top_invalid_str = "|".join(f"{w}:{c}" for w, c in Counter(invalid_words).most_common(top_n))

    return {
        "total_words": total_words,
        "valid_words": valid_count,
        "invalid_words": len(invalid_words),
        "lexicality": lexicality,
        "avg_zipf": avg_zipf,
        "top_invalid": top_invalid_str,
    }


In [ ]:
def stream_data(batch_size=1000):
    """Lit la base par blocs pour éviter de tout charger en mémoire."""
    con = duckdb.connect("../data/ocr-fulltext.db")
    offset = 0
    while True:
        batch = con.sql(f"""
            SELECT id, genre, try(decode(content)) AS content_text
            FROM fulltext_ocr_generated
            LIMIT {batch_size} OFFSET {offset}
        """).fetchall()
        if not batch:
            break
        yield batch
        offset += batch_size


In [ ]:
# ── Multiprocessing avec fastText dans chaque worker ──────────────────
#
# Pourquoi _init_worker ?
# Pool() crée des sous-processus séparés qui ne voient pas les variables
# du notebook (comme lang_model). L'initializer charge le modèle une
# seule fois dans chaque worker au démarrage, ce qui est bien plus
# efficace que de le recharger à chaque ligne.
#
# Pourquoi les imports dans _init_worker ?
# Chaque worker est un processus Python vierge — les imports du notebook
# ne sont pas transmis. Il faut tout réimporter dans le worker.
#
# Pourquoi le monkey-patch numpy ?
# fastText utilise np.array(..., copy=False) incompatible avec NumPy 2.x.
# On remplace predict() par une version qui utilise np.asarray().

_worker_lang_model = None

def _init_worker(model_path):
    global _worker_lang_model
    import fasttext
    import numpy as np

    # Patch numpy directement : on remplace array par asarray quand copy=False
    _orig_array = np.array
    def _patched_array(obj, *args, copy=None, **kwargs):
        if copy is False:
            return np.asarray(obj, *args, **kwargs)
        return _orig_array(obj, *args, copy=copy, **kwargs)
    np.array = _patched_array

    _worker_lang_model = fasttext.load_model(model_path)

def process_row(row, threshold=1, lang="en"):
    doc_id, genre, content = row
    if content is None:
        return None

    # Décodage BLOB → str
    if isinstance(content, bytes):
        try:
            text = content.decode("utf-8", errors="ignore")
        except Exception:
            text = content.decode("latin-1", errors="ignore")
    else:
        text = str(content)

    if not text or len(text.strip()) < 20:
        return None

    # Détection langue (modèle chargé localement dans le worker)
    clean = text.replace("\n", " ").strip()
    labels, probs = _worker_lang_model.predict(clean, k=1)
    lang_detected = labels[0].replace("__label__", "")
    prob = float(probs[0])

    # Filtre : anglais uniquement
    if lang_detected != "en":
        return None

    analysis = analyze_text(text, threshold, lang)
    return {
        "id": doc_id,
        "genre": genre,
        "lang": lang_detected,
        "lang_prob": prob,
        **analysis
    }

from tqdm import tqdm

def process_database_parallel(batch_size=1000, threshold=1, lang="en"):
    n_workers = max(cpu_count() - 1, 1)
    print(f"Workers utilisés: {n_workers}")
    results = []
    with Pool(n_workers, initializer=_init_worker, initargs=(MODEL_PATH,)) as pool:
        for batch in tqdm(stream_data(batch_size), desc="Batches traités", unit="batch"):
            processed = pool.starmap(process_row, [(row, threshold, lang) for row in batch])
            results.extend([r for r in processed if r is not None])
    return results


In [ ]:
def compute_global_summary(report):
    total_words = sum(r["total_words"] for r in report)
    total_valid = sum(r["valid_words"] for r in report)
    lexicality = (total_valid / total_words * 100) if total_words else 0
    global_invalid = Counter()
    for r in report:
        if r.get("top_invalid"):
            for item in r["top_invalid"].split("|"):
                w, c = item.split(":")
                global_invalid[w] += int(c)
    return {
        "total_docs": len(report),
        "total_words": total_words,
        "lexicality": lexicality,
        "top_invalid_global": global_invalid.most_common(50),
    }

def export_report_to_csv(report, output_path="report.csv"):
    if not report:
        return
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=report[0].keys())
        writer.writeheader()
        writer.writerows(report)


In [ ]:
# Lance tout le pipeline
BATCH_SIZE = 1000
THRESHOLD  = 1
LANG       = "en"

report = process_database_parallel(batch_size=BATCH_SIZE, threshold=THRESHOLD, lang=LANG)
report = [r for r in report if r and r["total_words"] > 0]

summary = compute_global_summary(report)
print("\n=== GLOBAL SUMMARY ===")
for k, v in summary.items():
    print(f"{k}: {v}")

export_report_to_csv(report, "LexicalityResults/ocr_lexicality-report-generated.csv")


In [ ]:
import fasttext
import numpy as np

# Patch numpy plutôt que fasttext
_orig_array = np.array
def _patched_array(obj, *args, copy=None, **kwargs):
    if copy is False:
        return np.asarray(obj, *args, **kwargs)
    return _orig_array(obj, *args, copy=copy, **kwargs)
np.array = _patched_array

lang_model = fasttext.load_model(MODEL_PATH)
print("lang_model chargé")

In [ ]:
import pandas as pd
non_english = []

for batch in stream_data(batch_size=5000):
    for row in batch:
        doc_id, genre, content = row
        if content is None:
            continue
        if isinstance(content, bytes):
            text = content.decode("utf-8", errors="ignore")
        else:
            text = str(content)
        if not text or len(text.strip()) < 20:
            continue
        clean = text.replace("\n", " ").strip()
        labels, probs = lang_model.predict(clean, k=1)
        lang_detected = labels[0].replace("__label__", "")
        if lang_detected != "en":
            non_english.append({"id": doc_id, "genre": genre, "lang": lang_detected, "lang_prob": float(probs[0])})

print(f"{len(non_english)} docs non anglais")
df_non_en = pd.DataFrame(non_english)
print(df_non_en["lang"].value_counts().head(20))
df_non_en.to_csv("LexicalityResults/non_english_docs-generated.csv", index=False)

## Analyse des résultats

In [ ]:
import pandas as pd

df = pd.read_csv("LexicalityResults/ocr_lexicality-report-generated.csv")

bins   = [0, 50, 70, 85, 95, 100]
labels = ["<50%", "50-70%", "70-85%", "85-95%", ">95%"]
df["quality_tier"] = pd.cut(df["lexicality"], bins=bins, labels=labels)

print(df["quality_tier"].value_counts().sort_index())
print(df.groupby("quality_tier", observed=True)["total_words"].mean())


In [ ]:
df["is_short"] = df["total_words"] < 500
print(df.groupby(["quality_tier", "is_short"], observed=True).size())


In [ ]:
# Docs prioritaires à retraiter : longs mais mauvaise lexicalité
priority = df[(df["total_words"] >= 500) & (df["lexicality"] < 50)]
print(f"{len(priority)} docs à retraiter en priorité")
priority[["id", "total_words", "lexicality", "avg_zipf"]].to_csv(
    "LexicalityResults/priority_reprocess-generated.csv", index=False)

borderline = df[(df["total_words"] >= 500) & (df["lexicality"].between(50, 70))]
print(f"{len(borderline)} docs borderline")


## Visualisations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import Normalize

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#cccccc", "axes.labelcolor": "#333333",
    "xtick.color": "#555555", "ytick.color": "#555555",
    "text.color": "#111111", "grid.color": "#e0e0e0",
    "font.family": "DejaVu Sans",
    "axes.spines.top": False, "axes.spines.right": False,
})

scores = df["lexicality"].dropna()
n = len(scores)
rng = np.random.default_rng(42)
x = rng.normal(0, 0.15, size=n)
cmap = plt.cm.viridis

fig, ax = plt.subplots(figsize=(12, 6))
sc = ax.scatter(x, scores, c=scores, cmap=cmap, alpha=0.25, s=6, linewidths=0, rasterized=True)

q25, median, q75 = scores.quantile([0.25, 0.50, 0.75])
mean_val = scores.mean()

ax.axhline(median,   color="#333333", linewidth=1.4, linestyle="-",  alpha=0.8)
ax.axhline(mean_val, color="#333333", linewidth=1.4, linestyle="--", alpha=0.8)
ax.text(0.02, median   + 0.8, f"Médiane {median:.1f} %",   transform=ax.get_yaxis_transform(), ha="left",   fontsize=9, color="#333333", va="bottom")
ax.text(0.50, mean_val + 0.8, f"Moyenne {mean_val:.1f} %", transform=ax.get_yaxis_transform(), ha="center", fontsize=9, color="#333333", va="bottom")
ax.text(0.98, q25 - 0.8, f"Q1 = {q25:.1f} %", transform=ax.get_yaxis_transform(), ha="right", fontsize=8, color="#666666", va="top")
ax.text(0.98, q75 + 0.8, f"Q3 = {q75:.1f} %", transform=ax.get_yaxis_transform(), ha="right", fontsize=8, color="#666666", va="bottom")

cbar = fig.colorbar(sc, ax=ax, pad=0.02, fraction=0.03)
cbar.set_label("Score de lexicalité (%)", fontsize=10)
ax.set_xlim(-0.6, 0.6)
ax.set_ylim(-2, 105)
ax.set_xticks([])
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f %%"))
ax.set_ylabel("Score de lexicalité", fontsize=11)
ax.set_title(f"Dispersion des scores de lexicalité\n(n = {n:,} documents — seuil Zipf ≥ 2.5)", fontsize=13, weight="bold", pad=14)
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig("Figures/fig_dispersion_lexicalite-generated.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
bins_w  = [0, 100, 500, 1_000, 2_500, 5_000, 10_000, 50_000, int(df["total_words"].max()) + 1]
labels_w = ["<100", "100-500", "500-1k", "1k-2.5k", "2.5k-5k", "5k-10k", "10k-50k", "50k+"]
df["word_bin"] = pd.cut(df["total_words"], bins=bins_w, labels=labels_w, right=False)

grp = df.groupby("word_bin", observed=True)["lexicality"].agg(
    median="median",
    q25=lambda s: s.quantile(0.25),
    q75=lambda s: s.quantile(0.75),
    count="count"
).reset_index()

xi = np.arange(len(grp))
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor("white")
ax.fill_between(xi, grp["q25"], grp["q75"], alpha=0.18, color=cmap(0.55), label="Intervalle Q1-Q3")
ax.plot(xi, grp["median"], marker="o", linewidth=2.5, color=cmap(0.75),
        markerfacecolor="white", markeredgecolor=cmap(0.75), markeredgewidth=2, markersize=8,
        label="Lexicalité médiane (%)", zorder=5)
for col in ["q25", "q75"]:
    ax.plot(xi, grp[col], linestyle="--", linewidth=1, color=cmap(0.45), alpha=0.5)

ax2 = ax.twinx()
ax2.bar(xi, grp["count"], color=cmap(0.3), alpha=0.18, width=0.6, zorder=0)
ax2.set_ylabel("Nombre de documents", fontsize=10, color="#555555")
ax2.tick_params(axis="y", labelcolor="#555555", labelsize=8)
ax2.spines["top"].set_visible(False)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))

for i, row in zip(xi, grp.itertuples()):
    ax.annotate(f"{row.median:.0f} %", xy=(i, row.median), xytext=(0, 9),
                textcoords="offset points", ha="center", fontsize=8, color="#333333", fontweight="bold")

ax.set_xticks(xi)
ax.set_xticklabels(grp["word_bin"], fontsize=9)
ax.set_xlabel("Nombre de mots par document", fontsize=11)
ax.set_ylabel("Score de lexicalité (%)", fontsize=11)
ax.set_ylim(0, 115)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f %%"))
ax.grid(axis="y", alpha=0.35, color="#dddddd")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_title("Lexicalité selon le nombre de mots\nMédiane par tranche — bande Q1/Q3 — volume en arrière-plan",
             fontsize=13, weight="bold", pad=14)
ax.legend(loc="lower right", fontsize=9, framealpha=0.7)
plt.tight_layout()
plt.savefig("Figures/fig_lexicalite_par_nb_mots-generated.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
df_clean = df.dropna(subset=["genre", "lexicality"])
df_clean = df_clean[df_clean["lexicality"] > 0]

df_grouped = (
    df_clean.groupby("genre")["lexicality"]
    .mean().reset_index()
    .sort_values(by="lexicality", ascending=False)
)

print(df_grouped)

plt.figure()
plt.bar(df_grouped["genre"], df_grouped["lexicality"])
plt.xlabel("Genre de document")
plt.ylabel("Lexicalité moyenne (%)")
plt.title("Lexicalité moyenne par genre")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.cm import get_cmap

# ── Chargement ────────────────────────────────────────────────────────
df = pd.read_csv("LexicalityResults/ocr_lexicality-report-generated.csv")
df = df.dropna(subset=["genre", "lexicality", "total_words"])
df = df[df["lexicality"] > 0]

# ── Agrégation par genre ───────────────────────────────────────────────
grp = (
    df.groupby("genre")
    .agg(
        lexicality_moy=("lexicality", "mean"),
        nb_mots_median=("total_words", "median"),
        nb_docs=("lexicality", "count"),
    )
    .reset_index()
    .sort_values("lexicality_moy", ascending=False)
)

# ── Figure ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor("#0f1117")
ax.set_facecolor("#0f1117")

# Colormap sur le nb de mots (plus c'est long, plus c'est chaud)
cmap = get_cmap("plasma")
norm = plt.Normalize(grp["nb_mots_median"].min(), grp["nb_mots_median"].max())
colors = [cmap(norm(v)) for v in grp["nb_mots_median"]]

# Barres
bars = ax.bar(
    grp["genre"],
    grp["lexicality_moy"],
    color=colors,
    width=0.6,
    zorder=2,
    edgecolor="none",
)

# Annotations : nb de docs + nb de mots médian sur chaque barre
for bar, (_, row) in zip(bars, grp.iterrows()):
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        h + 0.4,
        f"{h:.1f}%",
        ha="center", va="bottom",
        fontsize=9, color="white", fontweight="bold",
    )
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        1.0,
        f"n={int(row['nb_docs']):,}\n~{int(row['nb_mots_median']):,} mots",
        ha="center", va="bottom",
        fontsize=7, color="#aaaaaa",
    )

# Colorbar (nb de mots médian)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.025)
cbar.set_label("Nb de mots médian par doc", fontsize=9, color="#cccccc")
cbar.ax.yaxis.set_tick_params(color="#cccccc", labelcolor="#cccccc")
cbar.outline.set_edgecolor("#333333")
cbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))

# ── Style ─────────────────────────────────────────────────────────────
ax.set_ylim(0, grp["lexicality_moy"].max() + 8)
ax.set_xlabel("Genre de document", fontsize=11, color="#cccccc", labelpad=10)
ax.set_ylabel("Lexicalité moyenne (%)", fontsize=11, color="#cccccc")
ax.set_title(
    "Lexicalité moyenne par genre\nCouleur = longueur médiane des documents",
    fontsize=14, fontweight="bold", color="white", pad=16,
)
ax.tick_params(axis="x", rotation=35, colors="#cccccc", labelsize=9)
ax.tick_params(axis="y", colors="#cccccc", labelsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f %%"))
ax.grid(axis="y", color="#2a2a3a", linewidth=0.8, zorder=0)
for spine in ax.spines.values():
    spine.set_edgecolor("#2a2a3a")

plt.tight_layout()
plt.savefig("Figures/fig_genre_lexicalite_mots.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Sauvegardé : Figures/fig_genre_lexicalite_mots.png")